In [1]:
# IMPORT LIBRARY
from pathlib import Path
import polars as pl
import duckdb

# MENENTUKAN FOLDER PROYEK
PROJECT_DIR = Path("/home/jovyan/work")

# LOKASI DATASET MENTAH
RAW_PATH = PROJECT_DIR / "data" / "raw" / "flood_dataset.parquet"

# LOKASI DATASET HASIL CLEANING
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
CLEAN_PATH = PROCESSED_DIR / "flood_dataset_cleaned.parquet"

# MEMASTIKAN FOLDER OUTPUT TERSEDIA
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# MEMERIKSA LOKASI FILE
print("=== PERSIAPAN DATA CLEANING ===")
print("Dataset mentah :", RAW_PATH)
print("File output    :", CLEAN_PATH)

if not RAW_PATH.exists():
    raise FileNotFoundError(
        f"Dataset tidak ditemukan: {RAW_PATH}"
    )

print("\nDataset mentah berhasil ditemukan.")
print("Folder processed siap digunakan.")

=== PERSIAPAN DATA CLEANING ===
Dataset mentah : /home/jovyan/work/data/raw/flood_dataset.parquet
File output    : /home/jovyan/work/data/processed/flood_dataset_cleaned.parquet

Dataset mentah berhasil ditemukan.
Folder processed siap digunakan.


In [2]:
# MEMBACA DATASET MENTAH
df_raw = pl.read_parquet(RAW_PATH)

print("=== INFORMASI DATASET MENTAH ===")
print(f"Jumlah baris : {df_raw.height:,}")
print(f"Jumlah kolom : {df_raw.width}")

print("\n=== TIPE DATA ===")
print(df_raw.schema)

print("\n=== 5 DATA PERTAMA ===")
print(df_raw.head(5))

=== INFORMASI DATASET MENTAH ===
Jumlah baris : 1,152,432
Jumlah kolom : 13

=== TIPE DATA ===
Schema({'district_id': String, 'date': Datetime(time_unit='us', time_zone=None), 'year': Int32, 'rain_1d': Float64, 'rain_3d': Float64, 'rain_7d': Float64, 'rain_30d': Float64, 'disc_now': Float64, 'disc_7d_mean': Float64, 'disc_change_3d': Float64, 'month': Int32, 'coastal': Int64, 'flood_label': Float64})

=== 5 DATA PERTAMA ===
shape: (5, 13)
┌───────────────┬──────────────┬──────┬─────────┬───┬──────────────┬───────┬─────────┬─────────────┐
│ district_id   ┆ date         ┆ year ┆ rain_1d ┆ … ┆ disc_change_ ┆ month ┆ coastal ┆ flood_label │
│ ---           ┆ ---          ┆ ---  ┆ ---     ┆   ┆ 3d           ┆ ---   ┆ ---     ┆ ---         │
│ str           ┆ datetime[μs] ┆ i32  ┆ f64     ┆   ┆ ---          ┆ i32   ┆ i64     ┆ f64         │
│               ┆              ┆      ┆         ┆   ┆ f64          ┆       ┆         ┆             │
╞═══════════════╪══════════════╪══════╪═════════╪═══

In [3]:
# 1. PROFILING DENGAN DUCKDB SUMMARIZE

# Membuat koneksi DuckDB
connection = duckdb.connect()

# Mendaftarkan DataFrame Polars ke DuckDB
connection.register("dataset", df_raw)

# Membuat ringkasan statistik seluruh kolom
profil = connection.execute("""
    SUMMARIZE SELECT * FROM dataset
""").pl()

# Menampilkan hasil profiling
print("=== HASIL DUCKDB SUMMARIZE ===")
print(profil)

print("\nJumlah kolom yang diprofiling:", profil.height)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== HASIL DUCKDB SUMMARIZE ===
shape: (13, 12)
┌────────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬─────────┬───────────┐
│ column_nam ┆ column_ty ┆ min       ┆ max       ┆ … ┆ q50       ┆ q75       ┆ count   ┆ null_perc │
│ e          ┆ pe        ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---     ┆ entage    │
│ ---        ┆ ---       ┆ str       ┆ str       ┆   ┆ str       ┆ str       ┆ i64     ┆ ---       │
│ str        ┆ str       ┆           ┆           ┆   ┆           ┆           ┆         ┆ decimal[9 │
│            ┆           ┆           ┆           ┆   ┆           ┆           ┆         ┆ ,2]       │
╞════════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═════════╪═══════════╡
│ district_i ┆ VARCHAR   ┆ IDN.10.13 ┆ IDN.9.9.9 ┆ … ┆ null      ┆ null      ┆ 1152432 ┆ 0.00      │
│ d          ┆           ┆ .10_1     ┆ _1        ┆   ┆           ┆           ┆         ┆           │
│ date       ┆ TIMESTAMP ┆ 2015-01-3 ┆ 2024-

In [4]:
# 2. DETEKSI MISSING VALUES

jumlah_baris = df_raw.height

null_summary = pl.DataFrame({
    "kolom": df_raw.columns,
    "jumlah_null": [
        df_raw[col].null_count()
        for col in df_raw.columns
    ]
}).with_columns(
    (
        pl.col("jumlah_null") / jumlah_baris * 100
    ).round(2).alias("persen_null")
)

print("=== DETEKSI MISSING VALUES ===")
print(null_summary)

print("\n=== TOTAL MISSING VALUES ===")
total_null = sum(null_summary["jumlah_null"])

print(f"Total baris       : {jumlah_baris:,}")
print(f"Total missing     : {total_null:,}")
print(f"Persentase missing: {total_null / (jumlah_baris * df_raw.width) * 100:.2f}%")

=== DETEKSI MISSING VALUES ===
shape: (13, 3)
┌────────────────┬─────────────┬─────────────┐
│ kolom          ┆ jumlah_null ┆ persen_null │
│ ---            ┆ ---         ┆ ---         │
│ str            ┆ i64         ┆ f64         │
╞════════════════╪═════════════╪═════════════╡
│ district_id    ┆ 0           ┆ 0.0         │
│ date           ┆ 0           ┆ 0.0         │
│ year           ┆ 0           ┆ 0.0         │
│ rain_1d        ┆ 0           ┆ 0.0         │
│ rain_3d        ┆ 0           ┆ 0.0         │
│ …              ┆ …           ┆ …           │
│ disc_7d_mean   ┆ 0           ┆ 0.0         │
│ disc_change_3d ┆ 0           ┆ 0.0         │
│ month          ┆ 0           ┆ 0.0         │
│ coastal        ┆ 0           ┆ 0.0         │
│ flood_label    ┆ 0           ┆ 0.0         │
└────────────────┴─────────────┴─────────────┘

=== TOTAL MISSING VALUES ===
Total baris       : 1,152,432
Total missing     : 0
Persentase missing: 0.00%


In [5]:
# 3. PENANGANAN MISSING VALUES

# Membuat salinan dataset mentah
df_missing_handled = df_raw.clone()

# Kolom pengukuran numerik:
# Jika terdapat null, gunakan median sebagai pengganti.
kolom_numerik = [
    "rain_1d",
    "rain_3d",
    "rain_7d",
    "rain_30d",
    "disc_now",
    "disc_7d_mean",
    "disc_change_3d"
]

for kolom in kolom_numerik:
    if df_missing_handled[kolom].null_count() > 0:
        median_value = df_missing_handled[kolom].median()

        df_missing_handled = df_missing_handled.with_columns(
            pl.col(kolom).fill_null(median_value)
        )

# Identitas wilayah yang kosong diberi kategori default.
if df_missing_handled["district_id"].null_count() > 0:
    df_missing_handled = df_missing_handled.with_columns(
        pl.col("district_id").fill_null("Tidak diketahui")
    )

# Tanggal yang kosong tidak diimputasi karena
# tanggal merupakan dasar analisis temporal.
df_missing_handled = df_missing_handled.filter(
    pl.col("date").is_not_null()
)

# Tahun dan bulan diturunkan kembali dari tanggal.
df_missing_handled = df_missing_handled.with_columns(
    pl.col("date").dt.year().cast(pl.Int32).alias("year"),
    pl.col("date").dt.month().cast(pl.Int32).alias("month")
)

# Label banjir dan indikator pesisir tidak diimputasi
# karena merupakan informasi kategori penting.
df_missing_handled = df_missing_handled.filter(
    pl.col("flood_label").is_not_null() &
    pl.col("coastal").is_not_null()
)

# Membandingkan hasil sebelum dan sesudah penanganan.
null_sebelum = sum(df_raw.null_count().row(0))
null_sesudah = sum(df_missing_handled.null_count().row(0))

print("=== HASIL PENANGANAN MISSING VALUES ===")
print(f"Jumlah baris awal     : {df_raw.height:,}")
print(f"Jumlah baris akhir    : {df_missing_handled.height:,}")
print(f"Missing sebelum       : {null_sebelum:,}")
print(f"Missing sesudah       : {null_sesudah:,}")
print(f"Baris dikeluarkan     : {df_raw.height - df_missing_handled.height:,}")

print("\n=== STATUS PENANGANAN ===")
if null_sebelum == 0:
    print("Tidak diperlukan imputasi karena seluruh kolom sudah lengkap.")
else:
    print("Penanganan missing values selesai.")

=== HASIL PENANGANAN MISSING VALUES ===
Jumlah baris awal     : 1,152,432
Jumlah baris akhir    : 1,152,432
Missing sebelum       : 0
Missing sesudah       : 0
Baris dikeluarkan     : 0

=== STATUS PENANGANAN ===
Tidak diperlukan imputasi karena seluruh kolom sudah lengkap.


In [6]:
# 4. DETEKSI OUTLIER MENGGUNAKAN METODE IQR

KOLOM_NUMERIK = [
    "rain_1d",
    "rain_3d",
    "rain_7d",
    "rain_30d",
    "disc_now",
    "disc_7d_mean",
    "disc_change_3d"
]

hasil_iqr = []

for kolom in KOLOM_NUMERIK:

    # Menghitung kuartil
    q1 = df_missing_handled[kolom].quantile(0.25)
    q3 = df_missing_handled[kolom].quantile(0.75)

    # Menghitung IQR
    iqr = q3 - q1

    # Menentukan batas outlier
    batas_bawah = q1 - (1.5 * iqr)
    batas_atas = q3 + (1.5 * iqr)

    # Menghitung jumlah outlier
    jumlah_outlier = df_missing_handled.filter(
        (pl.col(kolom) < batas_bawah) |
        (pl.col(kolom) > batas_atas)
    ).height

    hasil_iqr.append({
        "kolom": kolom,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "batas_bawah": batas_bawah,
        "batas_atas": batas_atas,
        "jumlah_outlier": jumlah_outlier,
        "persentase_outlier": (
            jumlah_outlier / df_missing_handled.height
        ) * 100
    })

outlier_summary = pl.DataFrame(hasil_iqr)

print("=== HASIL DETEKSI OUTLIER METODE IQR ===")
print(outlier_summary)

=== HASIL DETEKSI OUTLIER METODE IQR ===
shape: (7, 8)
┌────────────┬──────────┬───────────┬───────────┬────────────┬────────────┬────────────┬───────────┐
│ kolom      ┆ Q1       ┆ Q3        ┆ IQR       ┆ batas_bawa ┆ batas_atas ┆ jumlah_out ┆ persentas │
│ ---        ┆ ---      ┆ ---       ┆ ---       ┆ h          ┆ ---        ┆ lier       ┆ e_outlier │
│ str        ┆ f64      ┆ f64       ┆ f64       ┆ ---        ┆ f64        ┆ ---        ┆ ---       │
│            ┆          ┆           ┆           ┆ f64        ┆            ┆ i64        ┆ f64       │
╞════════════╪══════════╪═══════════╪═══════════╪════════════╪════════════╪════════════╪═══════════╡
│ rain_1d    ┆ 0.3      ┆ 9.8       ┆ 9.5       ┆ -13.95     ┆ 24.05      ┆ 60717      ┆ 5.268597  │
│ rain_3d    ┆ 2.2      ┆ 30.9      ┆ 28.7      ┆ -40.85     ┆ 73.95      ┆ 34406      ┆ 2.985512  │
│ rain_7d    ┆ 8.4      ┆ 72.2      ┆ 63.8      ┆ -87.3      ┆ 167.9      ┆ 23955      ┆ 2.078648  │
│ rain_30d   ┆ 60.0     ┆ 310.2     

In [7]:
# 4. PENANGANAN OUTLIER DENGAN WINSORIZATION

# Menyiapkan daftar kolom hasil winsorization
winsor_expressions = []
hasil_winsor = []

for kolom in KOLOM_NUMERIK:

    # Menghitung Q1 dan Q3
    q1 = df_missing_handled[kolom].quantile(0.25)
    q3 = df_missing_handled[kolom].quantile(0.75)

    # Menghitung IQR dan batas outlier
    iqr = q3 - q1
    batas_bawah = q1 - (1.5 * iqr)
    batas_atas = q3 + (1.5 * iqr)

    # Menghitung jumlah nilai yang akan dibatasi
    jumlah_diclip = df_missing_handled.filter(
        (pl.col(kolom) < batas_bawah) |
        (pl.col(kolom) > batas_atas)
    ).height

    # Winsorization pada kolom tambahan
    winsor_expressions.append(
        pl.col(kolom)
        .clip(
            lower_bound=float(batas_bawah),
            upper_bound=float(batas_atas)
        )
        .alias(f"{kolom}_winsorized")
    )

    hasil_winsor.append({
        "kolom": kolom,
        "batas_bawah": batas_bawah,
        "batas_atas": batas_atas,
        "jumlah_diclip": jumlah_diclip
    })

# Membuat dataset dengan kolom hasil winsorization
df_outlier_handled = df_missing_handled.with_columns(
    winsor_expressions
)

print("=== HASIL WINSORIZATION ===")
print(pl.DataFrame(hasil_winsor))

print("\n=== VALIDASI HASIL ===")
print(f"Baris sebelum  : {df_missing_handled.height:,}")
print(f"Baris sesudah  : {df_outlier_handled.height:,}")
print(f"Kolom awal     : {df_missing_handled.width}")
print(f"Kolom sekarang : {df_outlier_handled.width}")

print("\nKolom asli tetap dipertahankan.")
print("Kolom winsorized berhasil ditambahkan.")

=== HASIL WINSORIZATION ===
shape: (7, 4)
┌────────────────┬─────────────┬────────────┬───────────────┐
│ kolom          ┆ batas_bawah ┆ batas_atas ┆ jumlah_diclip │
│ ---            ┆ ---         ┆ ---        ┆ ---           │
│ str            ┆ f64         ┆ f64        ┆ i64           │
╞════════════════╪═════════════╪════════════╪═══════════════╡
│ rain_1d        ┆ -13.95      ┆ 24.05      ┆ 60717         │
│ rain_3d        ┆ -40.85      ┆ 73.95      ┆ 34406         │
│ rain_7d        ┆ -87.3       ┆ 167.9      ┆ 23955         │
│ rain_30d       ┆ -315.3      ┆ 685.5      ┆ 10907         │
│ disc_now       ┆ -14.16      ┆ 25.52      ┆ 159404        │
│ disc_7d_mean   ┆ -14.305     ┆ 25.883571  ┆ 158128        │
│ disc_change_3d ┆ -2.015      ┆ 1.705      ┆ 302722        │
└────────────────┴─────────────┴────────────┴───────────────┘

=== VALIDASI HASIL ===
Baris sebelum  : 1,152,432
Baris sesudah  : 1,152,432
Kolom awal     : 13
Kolom sekarang : 20

Kolom asli tetap dipertahankan.
K

In [8]:
# 5. DEDUPLIKASI BERDASARKAN KUNCI BISNIS

KOLOM_KUNCI = ["district_id", "date"]

# Menghitung jumlah data sebelum deduplikasi
total_baris = df_outlier_handled.height

# Menghitung jumlah kombinasi kunci yang unik
baris_unik = df_outlier_handled.select(
    pl.struct(KOLOM_KUNCI).n_unique()
).item()

print("=== PEMERIKSAAN DUPLIKASI ===")
print(f"Total baris                  : {total_baris:,}")
print(f"Baris unik berdasarkan kunci: {baris_unik:,}")
print(f"Duplikasi ditemukan          : {total_baris - baris_unik:,}")

# Melakukan deduplikasi berdasarkan kunci bisnis
df_dedup = df_outlier_handled.unique(
    subset=KOLOM_KUNCI,
    keep="first",
    maintain_order=True
)

# Validasi hasil deduplikasi
print("\n=== HASIL DEDUPLIKASI ===")
print(f"Baris sebelum deduplikasi : {total_baris:,}")
print(f"Baris setelah deduplikasi: {df_dedup.height:,}")
print(f"Baris yang dihapus       : {total_baris - df_dedup.height:,}")

print("\n=== VALIDASI KUNCI BISNIS ===")

duplikat_akhir = (
    df_dedup
    .group_by(KOLOM_KUNCI)
    .len()
    .filter(pl.col("len") > 1)
    .height
)

print(f"Duplikasi tersisa: {duplikat_akhir:,}")

=== PEMERIKSAAN DUPLIKASI ===
Total baris                  : 1,152,432
Baris unik berdasarkan kunci: 1,152,432
Duplikasi ditemukan          : 0

=== HASIL DEDUPLIKASI ===
Baris sebelum deduplikasi : 1,152,432
Baris setelah deduplikasi: 1,152,432
Baris yang dihapus       : 0

=== VALIDASI KUNCI BISNIS ===
Duplikasi tersisa: 0


In [9]:
# 6. VALIDASI AKHIR DATASET

print("=== VALIDASI AKHIR DATASET ===")

# 1. Pemeriksaan missing values
total_missing = sum(df_dedup.null_count().row(0))

# 2. Pemeriksaan duplikasi kunci bisnis
duplicate_keys = (
    df_dedup
    .group_by(["district_id", "date"])
    .len()
    .filter(pl.col("len") > 1)
    .height
)

# 3. Pemeriksaan kesesuaian tahun dan tanggal
invalid_year = df_dedup.filter(
    pl.col("year") != pl.col("date").dt.year()
).height

# 4. Pemeriksaan kesesuaian bulan dan tanggal
invalid_month = df_dedup.filter(
    pl.col("month") != pl.col("date").dt.month()
).height

# 5. Pemeriksaan label banjir
invalid_label = df_dedup.filter(
    ~pl.col("flood_label").is_in([0.0, 1.0])
).height

# 6. Pemeriksaan kategori pesisir
invalid_coastal = df_dedup.filter(
    ~pl.col("coastal").is_in([0, 1])
).height

# 7. Pemeriksaan nilai non-finite
kolom_float_final = KOLOM_NUMERIK + [
    f"{kolom}_winsorized" for kolom in KOLOM_NUMERIK
]

total_non_finite = sum(
    df_dedup.filter(
        ~pl.col(kolom).is_finite()
    ).height
    for kolom in kolom_float_final
)

# Menampilkan hasil
hasil_validasi = pl.DataFrame({
    "Pemeriksaan": [
        "Missing values",
        "Duplikasi kunci bisnis",
        "Tahun tidak sesuai tanggal",
        "Bulan tidak sesuai tanggal",
        "Label banjir tidak valid",
        "Kategori pesisir tidak valid",
        "Nilai non-finite"
    ],
    "Jumlah masalah": [
        total_missing,
        duplicate_keys,
        invalid_year,
        invalid_month,
        invalid_label,
        invalid_coastal,
        total_non_finite
    ]
})

print(hasil_validasi)

print("\n=== RINGKASAN DATASET ===")
print(f"Jumlah baris : {df_dedup.height:,}")
print(f"Jumlah kolom : {df_dedup.width}")

if sum(hasil_validasi["Jumlah masalah"]) == 0:
    print("\nSTATUS: SELURUH VALIDASI BERHASIL")
else:
    print("\nSTATUS: MASIH ADA MASALAH YANG PERLU DIPERIKSA")

=== VALIDASI AKHIR DATASET ===
shape: (7, 2)
┌──────────────────────────────┬────────────────┐
│ Pemeriksaan                  ┆ Jumlah masalah │
│ ---                          ┆ ---            │
│ str                          ┆ i64            │
╞══════════════════════════════╪════════════════╡
│ Missing values               ┆ 0              │
│ Duplikasi kunci bisnis       ┆ 0              │
│ Tahun tidak sesuai tanggal   ┆ 0              │
│ Bulan tidak sesuai tanggal   ┆ 0              │
│ Label banjir tidak valid     ┆ 0              │
│ Kategori pesisir tidak valid ┆ 0              │
│ Nilai non-finite             ┆ 0              │
└──────────────────────────────┴────────────────┘

=== RINGKASAN DATASET ===
Jumlah baris : 1,152,432
Jumlah kolom : 20

STATUS: SELURUH VALIDASI BERHASIL


In [10]:
# 7. MENYIMPAN DATASET HASIL CLEANING

# Mengurutkan data berdasarkan kecamatan dan tanggal
df_final = df_dedup.sort(["district_id", "date"])

# Menyimpan dataset dalam format Parquet
df_final.write_parquet(
    CLEAN_PATH,
    compression="zstd"
)

# Membaca ulang file untuk memastikan penyimpanan berhasil
df_saved = pl.read_parquet(CLEAN_PATH)

print("=== PENYIMPANAN DATASET FINAL ===")
print(f"Lokasi file    : {CLEAN_PATH}")
print(f"Ukuran file    : {CLEAN_PATH.stat().st_size / (1024**2):.2f} MB")
print(f"Jumlah baris   : {df_saved.height:,}")
print(f"Jumlah kolom   : {df_saved.width}")
print(f"File tersedia  : {CLEAN_PATH.exists()}")

# Verifikasi hasil penyimpanan
assert df_saved.height == df_final.height
assert df_saved.width == df_final.width
assert sum(df_saved.null_count().row(0)) == 0

print("\nSTATUS: DATASET FINAL BERHASIL DISIMPAN DAN DIVERIFIKASI.")

=== PENYIMPANAN DATASET FINAL ===
Lokasi file    : /home/jovyan/work/data/processed/flood_dataset_cleaned.parquet
Ukuran file    : 27.00 MB
Jumlah baris   : 1,152,432
Jumlah kolom   : 20
File tersedia  : True

STATUS: DATASET FINAL BERHASIL DISIMPAN DAN DIVERIFIKASI.


In [11]:
# PROFILING ULANG DATASET FINAL DENGAN DUCKDB

con_final = duckdb.connect()

final_path = CLEAN_PATH.as_posix()

summary_final = con_final.execute(f"""
    SUMMARIZE
    SELECT *
    FROM read_parquet('{final_path}')
""").df()

print("=== DUCKDB SUMMARIZE DATASET FINAL ===")
print(summary_final.to_string(index=False))

# Menyimpan hasil SUMMARIZE terbaru
SUMMARY_PATH = PROCESSED_DIR / "duckdb_summarize.csv"

summary_final.to_csv(SUMMARY_PATH, index=False)

print("\n=== HASIL PENYIMPANAN ===")
print(f"Jumlah kolom: {len(summary_final)}")
print(f"File tersimpan: {SUMMARY_PATH.exists()}")

con_final.close()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== DUCKDB SUMMARIZE DATASET FINAL ===
              column_name column_type                 min                 max  approx_unique                   avg                std                        q25                        q50                        q75   count  null_percentage
              district_id     VARCHAR      IDN.10.13.10_1         IDN.9.9.9_1            347                   NaN                NaN                        NaN                        NaN                        NaN 1152432              0.0
                     date   TIMESTAMP 2015-01-30 00:00:00 2024-12-31 00:00:00           4543   2020-01-15 12:00:00                NaN 2017-07-23 11:19:09.623087 2020-01-14 22:35:50.230549 2022-07-08 01:18:24.737138 1152432              0.0
                     year     INTEGER                2015                2024             11    2019.5364238410596 2.8556717940747824                       2017                       2020                       2022 1152432              0.0
 

In [12]:
# PEMERIKSAAN VALIDITAS NILAI

print("=== PEMERIKSAAN LABEL BANJIR ===")
print(df_raw["flood_label"].value_counts().sort("flood_label"))

invalid_label = df_raw.filter(
    ~pl.col("flood_label").is_in([0.0, 1.0])
).height

print(f"Label banjir tidak valid: {invalid_label:,}")


print("\n=== PEMERIKSAAN WILAYAH PESISIR ===")
print(df_raw["coastal"].value_counts().sort("coastal"))

invalid_coastal = df_raw.filter(
    ~pl.col("coastal").is_in([0, 1])
).height

print(f"Nilai coastal tidak valid: {invalid_coastal:,}")


print("\n=== PEMERIKSAAN IDENTITAS WILAYAH ===")
district_kosong = df_raw.filter(
    pl.col("district_id").str.strip_chars() == ""
).height

print(f"Identitas wilayah kosong: {district_kosong:,}")


# PEMERIKSAAN NILAI NUMERIK TIDAK TERBATAS
kolom_float = [
    "rain_1d",
    "rain_3d",
    "rain_7d",
    "rain_30d",
    "disc_now",
    "disc_7d_mean",
    "disc_change_3d",
    "flood_label"
]

print("\n=== PEMERIKSAAN NILAI NON-FINITE ===")

for kolom in kolom_float:
    jumlah = df_raw.filter(
        ~pl.col(kolom).is_finite()
    ).height

    print(f"{kolom}: {jumlah:,}")

=== PEMERIKSAAN LABEL BANJIR ===
shape: (2, 2)
┌─────────────┬─────────┐
│ flood_label ┆ count   │
│ ---         ┆ ---     │
│ f64         ┆ u32     │
╞═════════════╪═════════╡
│ 0.0         ┆ 1068121 │
│ 1.0         ┆ 84311   │
└─────────────┴─────────┘
Label banjir tidak valid: 0

=== PEMERIKSAAN WILAYAH PESISIR ===
shape: (2, 2)
┌─────────┬─────────┐
│ coastal ┆ count   │
│ ---     ┆ ---     │
│ i64     ┆ u32     │
╞═════════╪═════════╡
│ 0       ┆ 115968  │
│ 1       ┆ 1036464 │
└─────────┴─────────┘
Nilai coastal tidak valid: 0

=== PEMERIKSAAN IDENTITAS WILAYAH ===
Identitas wilayah kosong: 0

=== PEMERIKSAAN NILAI NON-FINITE ===
rain_1d: 0
rain_3d: 0
rain_7d: 0
rain_30d: 0
disc_now: 0
disc_7d_mean: 0
disc_change_3d: 0
flood_label: 0


In [13]:
# PROSES DATA CLEANING

kolom_numerik = [
    "rain_1d",
    "rain_3d",
    "rain_7d",
    "rain_30d",
    "disc_now",
    "disc_7d_mean",
    "disc_change_3d",
    "flood_label"
]

# Membersihkan spasi pada identitas wilayah
df_clean = df_raw.with_columns(
    pl.col("district_id").str.strip_chars()
)

# Menentukan aturan validasi
valid_checks = [
    pl.col("district_id").is_not_null(),
    pl.col("district_id") != "",
    pl.col("date").is_not_null(),
    pl.col("year") == pl.col("date").dt.year(),
    pl.col("month") == pl.col("date").dt.month(),
    pl.col("month").is_between(1, 12),
    pl.col("flood_label").is_in([0.0, 1.0]),
    pl.col("coastal").is_in([0, 1])
]

# Memastikan seluruh nilai numerik finite
valid_checks.extend([
    pl.col(kolom).is_finite()
    for kolom in kolom_numerik
])

# Mempertahankan baris yang memenuhi seluruh aturan
df_clean = df_clean.filter(
    pl.all_horizontal(valid_checks)
)

# Mengurutkan data berdasarkan wilayah dan tanggal
df_clean = df_clean.sort(["district_id", "date"])

# Menyimpan hasil cleaning
df_clean.write_parquet(
    CLEAN_PATH,
    compression="zstd"
)

# Ringkasan hasil cleaning
print("=== HASIL DATA CLEANING ===")
print(f"Baris sebelum cleaning : {df_raw.height:,}")
print(f"Baris setelah cleaning : {df_clean.height:,}")
print(f"Baris yang dikeluarkan : {df_raw.height - df_clean.height:,}")
print(f"Jumlah kolom           : {df_clean.width}")
print(f"Ukuran file hasil      : {CLEAN_PATH.stat().st_size / (1024**2):.2f} MB")
print(f"File berhasil disimpan : {CLEAN_PATH.exists()}")

=== HASIL DATA CLEANING ===
Baris sebelum cleaning : 1,152,432
Baris setelah cleaning : 1,152,432
Baris yang dikeluarkan : 0
Jumlah kolom           : 13
Ukuran file hasil      : 14.53 MB
File berhasil disimpan : True


In [14]:
# VALIDASI DAN PROFILING MENGGUNAKAN DUCKDB

con = duckdb.connect()

CLEAN_PATH_SQL = CLEAN_PATH.as_posix()

# Menghitung jumlah baris hasil cleaning
jumlah_baris_duckdb = con.execute(f"""
    SELECT COUNT(*)
    FROM read_parquet('{CLEAN_PATH_SQL}')
""").fetchone()[0]

print("=== VALIDASI DATA CLEANING ===")
print(f"Jumlah baris Polars : {df_clean.height:,}")
print(f"Jumlah baris DuckDB : {jumlah_baris_duckdb:,}")

# DuckDB SUMMARIZE
print("\n=== DUCKDB SUMMARIZE ===")

summary = con.execute(f"""
    SUMMARIZE
    SELECT *
    FROM read_parquet('{CLEAN_PATH_SQL}')
""").df()

print(summary.to_string(index=False))

con.close()

=== VALIDASI DATA CLEANING ===
Jumlah baris Polars : 1,152,432
Jumlah baris DuckDB : 1,152,432

=== DUCKDB SUMMARIZE ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   column_name column_type                 min                 max  approx_unique                  avg                 std                        q25                        q50                        q75   count  null_percentage
   district_id     VARCHAR      IDN.10.13.10_1         IDN.9.9.9_1            347                  NaN                 NaN                        NaN                        NaN                        NaN 1152432              0.0
          date   TIMESTAMP 2015-01-30 00:00:00 2024-12-31 00:00:00           4543  2020-01-15 12:00:00                 NaN 2017-07-21 03:59:39.940467 2020-01-15 22:08:38.738592 2022-07-11 01:45:51.158573 1152432              0.0
          year     INTEGER                2015                2024             11   2019.5364238410596  2.8556717940748193                       2017                       2020                       2022 1152432              0.0
       rain_1d      DOUBLE                 0.0               258.7           1546   

In [15]:
# VALIDASI AKHIR DATASET HASIL CLEANING

df_check = pl.read_parquet(CLEAN_PATH)

print("=== VALIDASI FILE HASIL CLEANING ===")

print(f"Jumlah baris       : {df_check.height:,}")
print(f"Jumlah kolom       : {df_check.width}")
print(f"Jumlah duplikat    : {df_check.is_duplicated().sum():,}")

total_missing = sum(df_check.null_count().row(0))
print(f"Total missing value: {total_missing:,}")

print("\n=== VALIDASI KUNCI WILAYAH DAN TANGGAL ===")

duplicate_key = (
    df_check.group_by(["district_id", "date"])
    .len()
    .filter(pl.col("len") > 1)
    .height
)

print(f"Duplikasi wilayah-tanggal: {duplicate_key:,}")

# MENYIMPAN HASIL DUCKDB SUMMARIZE
SUMMARY_PATH = PROCESSED_DIR / "duckdb_summarize.csv"

summary.to_csv(SUMMARY_PATH, index=False)

print("\n=== PENYIMPANAN LAPORAN ===")
print(f"File SUMMARIZE tersimpan: {SUMMARY_PATH.exists()}")
print(f"Lokasi file: {SUMMARY_PATH}")

=== VALIDASI FILE HASIL CLEANING ===
Jumlah baris       : 1,152,432
Jumlah kolom       : 13
Jumlah duplikat    : 0
Total missing value: 0

=== VALIDASI KUNCI WILAYAH DAN TANGGAL ===
Duplikasi wilayah-tanggal: 0

=== PENYIMPANAN LAPORAN ===
File SUMMARIZE tersimpan: True
Lokasi file: /home/jovyan/work/data/processed/duckdb_summarize.csv


## Catatan Milestone 2

### 1. Missing Values
Hasil pemeriksaan menunjukkan tidak terdapat missing values
pada seluruh 13 kolom dataset. Oleh karena itu, tidak diperlukan
proses imputasi dan seluruh baris dipertahankan.

### 2. Outlier
Deteksi outlier dilakukan menggunakan metode Interquartile Range
(IQR) dengan batas 1,5 × IQR.

Penanganan dilakukan menggunakan Winsorization. Nilai asli tetap
dipertahankan, sedangkan hasil pembatasan disimpan dalam tujuh
kolom tambahan dengan akhiran `_winsorized`.

### 3. Deduplikasi
Deduplikasi dilakukan berdasarkan kunci bisnis `district_id`
dan `date`. Tidak ditemukan duplikasi sehingga tidak ada baris
yang dihapus.

### 4. Validasi
Validasi akhir menunjukkan:
- Missing values: 0
- Duplikasi kunci bisnis: 0
- Ketidaksesuaian tahun dan tanggal: 0
- Ketidaksesuaian bulan dan tanggal: 0
- Label banjir tidak valid: 0
- Kategori pesisir tidak valid: 0
- Nilai non-finite: 0

### 5. Hasil Akhir
Dataset hasil cleaning berisi 1.152.432 baris dan 20 kolom.
Dataset disimpan dalam format Parquet dengan kompresi Zstandard.
Profiling akhir dilakukan menggunakan DuckDB SUMMARIZE.